### Configuração Inicial e Bibliotecas:

-

In [ ]:
# import tensorflow_datasets as tfds
from attention_graph_util import *
import seaborn as sns
import matplotlib as mpl

import torch
import matplotlib.pyplot as plt
rc={'font.size': 10, 'axes.labelsize': 10, 'legend.fontsize': 10.0, 
    'axes.titlesize': 32, 'xtick.labelsize': 20, 'ytick.labelsize': 16}
plt.rcParams.update(**rc)
mpl.rcParams['axes.linewidth'] = .5 #set the value globally


# Importando apenas o que importa, sem o "*"
from transformers import (
    AutoModelForCausalLM, AutoTokenizer
)

from pathlib import Path

In [ ]:
def plot_attention_heatmap(att, s_position, t_positions, tokens_list):
    # Slice usando PyTorch
    cls_att = torch.flip(att[:, s_position, t_positions], dims=[0]).clone()
    
    cls_att[:, s_position] = 0.0 
    
    xticklb = [tokens_list[i] for i in t_positions]
    yticklb = [str(i) if i % 2 == 0 else '' for i in torch.arange(att.shape[0], 0, -1)]
    
    ax = sns.heatmap(cls_att.cpu().numpy(), xticklabels=xticklb, yticklabels=yticklb, cmap="YlOrRd")
    return ax

In [ ]:
def convert_adjmat_tomats(adjmat, n_layers, l):
    mats = torch.zeros((n_layers, l, l))
    
    for i in range(n_layers):
        mats[i] = adjmat[(i+1)*l:(i+2)*l, i*l:(i+1)*l]
        
    return mats

## Carregando o Modelo e Escolhendo um Exemplo

In [ ]:
pretrained_weights = 'gpt2'
model_id = pretrained_weights.split("/")[-1]
family = 'gpt'
print(f"Model: {model_id}, Family: {family}")

model = AutoModelForCausalLM.from_pretrained(
    pretrained_weights,
    output_hidden_states=True,
    output_attentions=True
)
model.zero_grad()
tokenizer = AutoTokenizer.from_pretrained(pretrained_weights, use_fast=True)

# Vamos analisar uma única frase como caso de estudo
sentence = "He talked to her about his book"
print(f"\nFrase em análise: '{sentence}'")

tokens =  tokenizer.tokenize(sentence)

# tokeniza a sentença
tf_input_ids = [tokenizer.eos_token_id] + tokenizer.encode(sentence)
tokens = [tokenizer.eos_token] + tokenizer.tokenize(sentence)

# transforma em um batch de sentença única
input_ids = torch.tensor([tf_input_ids])


## Inferência e Extração das Atenções (Raw Attention)

In [ ]:
model_outputs = model(input_ids=input_ids)

# Extrai as atenções
all_attentions = model_outputs['attentions'] 
_attentions = [att.detach() for att in all_attentions] 
attentions_mat = torch.stack(_attentions)[:, 0]  # Remove dimensão do batch

print(f"Formato final da matriz de atenção (Camadas, Cabeças, Seq, Seq): {attentions_mat.shape}")

# Predição do próximo token
output = model_outputs.logits[0]
predicted_target = torch.nn.Softmax(dim=0)(output[-1, :])
previewd = torch.argmax(predicted_target, dim=-1)

print(f"Próximo token adivinhado: {tokenizer.decode(previewd.item())}")

# Plot da Raw Attention Média (focando na última palavra)
plt.figure(figsize=(3,6))
t_pos = tuple(range(len(tokens)))

raw_att_norm = attentions_mat.sum(dim=1) / attentions_mat.shape[1]

plot_attention_heatmap(
    raw_att_norm,
    s_position=len(tf_input_ids) - 1,
    t_positions=t_pos,
    tokens_list=tokens
)

plt.title("Raw Attention Média")
plt.show()

## O Grafo e o Attention Flow

In [ ]:
# Preparação da matriz (Conexões residuais)
res_att_mat = attentions_mat.sum(dim=1) / attentions_mat.shape[1]

res_att_mat = res_att_mat + 0.5 * torch.eye(res_att_mat.shape[1])[None, ...]
res_att_mat = res_att_mat / res_att_mat.sum(dim=-1)[..., None]

# Construção do Grafo
res_adj_mat, res_labels_to_index = get_adjmat(mat=res_att_mat, input_tokens=tokens)

res_G = draw_attention_graph(
    res_adj_mat,
    res_labels_to_index,
    n_layers=res_att_mat.shape[0],
    length=res_att_mat.shape[-1]
)

plt.title("Grafo de Atenção Base")
plt.show()

# Calculando o Attention Flow
last_layer_name = f'L{attentions_mat.shape[0]}'
output_nodes = [key for key in res_labels_to_index if last_layer_name in key]
input_nodes = [key for key, idx in res_labels_to_index.items() if idx < attentions_mat.shape[-1]]

flow_values = compute_flows(res_G, res_labels_to_index, input_nodes, length=attentions_mat.shape[-1])

flow_att_mat = convert_adjmat_tomats(
    flow_values,
    n_layers=attentions_mat.shape[0],
    l=attentions_mat.shape[-1]
)

plt.figure(figsize=(3,6))
plot_attention_heatmap(
    flow_att_mat,
    s_position=len(tf_input_ids) - 1,
    t_positions=t_pos,
    tokens_list=tokens
)

plt.title("Attention Flow")
plt.show()

## Attention Rollout

In [ ]:
# CÉLULA 5
joint_attentions = compute_joint_attention(res_att_mat, add_residual=False)

plt.figure(figsize=(3,6))
plot_attention_heatmap(joint_attentions, s_position=len(tf_input_ids)-1, t_positions=t_pos, tokens_list=tokens)
plt.title("Attention Rollout")
plt.show()

## Loop com mais exemplos

In [ ]:
sentences2 = {}

sentences2[0] = "He talked to her about his book"
sentences2[1] = "She asked the doctor about her"
sentences2[2] = "The author talked to Sara about his"
sentences2[3] = "John tried to convince Mary of his love and brought flowers for "
sentences2[4] = "Mary convinced John of her"
sentences2[5] = "Barack Obama was the president of the"
sentences2[6] = "Artificial intelligence is the field of study that"
sentences2[7] = "Why is the sky blue?"
sentences2[8] = "the capital of France is"

for ex_id in range(len(sentences)):
    
    OUTPUT_DIR = IMAGES_DIR / str(ex_id)
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    sentence = sentences[ex_id]

    tokens = tokenizer.tokenize(sentence)

    # tokeniza a sentença
    tf_input_ids = [tokenizer.eos_token_id] + tokenizer.encode(sentence)
    tokens = [tokenizer.eos_token] + tokenizer.tokenize(sentence)

    # transforma em batch
    input_ids = torch.tensor([tf_input_ids])

    # saída do modelo
    model_outputs = model(input_ids=input_ids)

    # hidden states e atenções
    all_hidden_states, all_attentions = model_outputs['hidden_states'], model_outputs['attentions']

    # transforma tudo em tensor PyTorch
    _attentions = [att.detach() for att in all_attentions]

    # 12 camadas, 1 batch, 12 cabeças, LXL
    attentions_mat = torch.stack(_attentions)[:, 0]

    # remove o token eos
    attentions_mat = attentions_mat[:, :, 1:, 1:]
    tokens = tokens[1:]
    tf_input_ids = tf_input_ids[1:]

    print(input_ids)
    print(tokens)

    # =========================
    # geração de texto
    # =========================
    max_l = 30
    output_ids = model.generate(input_ids, max_length=max_l)

    print(f"Próximos {max_l} tokens gerados pelo modelo:\n{tokenizer.decode(output_ids[0], skip_special_tokens=True)}")

    # logits
    output = model(input_ids).logits[0]

    predicted_target = torch.nn.Softmax(dim=0)(output[-1, :])
    previewd = torch.argmax(predicted_target, dim=-1)

    print(f"Próximo token gerado pelo modelo:\n{tokenizer.decode(previewd.item())}")

    # =========================
    # Top-k
    # =========================
    k = 5
    topk_vals, topk_idx = torch.topk(predicted_target, k)

    print(f"Top {k} tokens:")
    for idx, val in zip(topk_idx, topk_vals):
        print(f"Token: {tokenizer.decode([idx.item()]):15} | prob: {val.item():.6f} | id: {idx.item()}")

    print("\n\n")

    # =========================
    # Barplot
    # =========================
    yax = [predicted_target[id].item() for id in topk_idx]
    xax = [tokenizer.decode([id.item()]) for id in topk_idx]

    fig = plt.figure(figsize=(6,6))
    ax = sns.barplot(x=xax, y=yax, linewidth=0)

    sns.despine(fig=fig, ax=None, top=True, right=True, left=True, bottom=False)

    ax.set_ylim(0,1)

    plt.savefig(OUTPUT_DIR / f'rat_gpt_bar_{ex_id}.png', format='png', transparent=True, dpi=360, bbox_inches='tight')
    plt.close()

    # =========================
    # Raw Attention
    # =========================
    plt.figure(figsize=(3,6))

    t_pos = tuple(range(len(tokens)))

    raw_att = attentions_mat.sum(dim=1) / attentions_mat.shape[1]

    plot_attention_heatmap(
        raw_att,
        s_position=len(tf_input_ids)-1,
        t_positions=t_pos,
        tokens_list=tokens
    )

    plt.savefig(OUTPUT_DIR / f'rat_gpt_att_{ex_id}.png', format='png', transparent=True, dpi=360, bbox_inches='tight')
    plt.close()

    # =========================
    # Residual Attention
    # =========================
    res_att_mat = attentions_mat.sum(dim=1) / attentions_mat.shape[1]

    res_att_mat = res_att_mat + torch.eye(res_att_mat.shape[1])[None, ...]
    res_att_mat = res_att_mat / res_att_mat.sum(dim=-1)[..., None]

    res_adj_mat, res_labels_to_index = get_adjmat(mat=res_att_mat, input_tokens=tokens)

    res_G = draw_attention_graph(
        res_adj_mat,
        res_labels_to_index,
        n_layers=res_att_mat.shape[0],
        length=res_att_mat.shape[-1]
    )

    # =========================
    # Attention Flow
    # =========================
    last_layer_name = f'L{attentions_mat.shape[0]}'

    output_nodes = []
    input_nodes = []

    for key in res_labels_to_index:
        if last_layer_name in key:
            output_nodes.append(key)
        if res_labels_to_index[key] < attentions_mat.shape[-1]:
            input_nodes.append(key)

    flow_values = compute_flows(res_G, res_labels_to_index, input_nodes, length=attentions_mat.shape[-1])

    flow_att_mat = convert_adjmat_tomats(
        flow_values,
        n_layers=attentions_mat.shape[0],
        l=attentions_mat.shape[-1]
    )

    plt.figure(figsize=(3,6))

    plot_attention_heatmap(
        flow_att_mat,
        s_position=len(tf_input_ids)-1,
        t_positions=t_pos,
        tokens_list=tokens
    )

    plt.savefig(OUTPUT_DIR / f'res_fat_gpt_att_{ex_id}.png', format='png', transparent=True, dpi=360, bbox_inches='tight')
    plt.close()

    # =========================
    # Attention Rollout
    # =========================
    joint_attentions = compute_joint_attention(res_att_mat, add_residual=False)

    plt.figure(figsize=(3,6))

    plot_attention_heatmap(
        joint_attentions,
        s_position=len(tf_input_ids)-1,
        t_positions=t_pos,
        tokens_list=tokens
    )

    plt.savefig(OUTPUT_DIR / f'res_jat_gpt_att_{ex_id}.png', format='png', transparent=True, dpi=360, bbox_inches='tight')
    plt.close()